# Peer University File Merge

Each peer university has two files:
- **Scopus file** — full feature set, no abstracts
- **Abstracts file** — real abstracts, fewer features

Both share an `EID` column. We left-join the abstracts onto the scopus file
so we keep all scopus features and fill in the `Abstract` column.

In [ ]:
import sys
sys.path.append('../')

import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)

## 1. File paths — verify before running

In [ ]:
# Each entry: (scopus_file, abstracts_file)
# NOTE: villanova scopus file was listed as 'villanova abstracts.csv' in the source —
#       assuming it should be 'villanova.csv'. Correct below if needed.
INSTITUTIONS = {
    'Lehigh':    ('../data/raw/lehigh.csv',             '../data/raw/lehigh abstracts.csv'),
    'Marquette': ('../data/raw/marquette.csv',           '../data/raw/marquette abstracts.csv'),
    'Villanova': ('../data/raw/villanova.csv',           '../data/raw/villanova abstracts.csv'),
}

print(f"{'Institution':<12} {'Scopus file':<40} {'Abstracts file':<40}")
print("-" * 94)
for inst, (scopus, abstracts) in INSTITUTIONS.items():
    s_ok = '✓' if Path(scopus).exists()    else '✗ MISSING'
    a_ok = '✓' if Path(abstracts).exists() else '✗ MISSING'
    print(f"{inst:<12} {s_ok+' '+scopus:<40} {a_ok+' '+abstracts:<40}")

## 2. Merge each institution: scopus (base) ← abstracts on EID

In [ ]:
def merge_institution(inst_name, scopus_path, abstracts_path):
    """Left-join abstract file onto scopus file on EID."""
    scopus = pd.read_csv(scopus_path, low_memory=False)
    abstracts = pd.read_csv(abstracts_path, low_memory=False)

    print(f"\n{'='*50}")
    print(f"{inst_name}")
    print(f"{'='*50}")
    print(f"  Scopus:    {scopus.shape[0]:,} rows × {scopus.shape[1]} cols")
    print(f"  Abstracts: {abstracts.shape[0]:,} rows × {abstracts.shape[1]} cols")

    # Confirm EID exists in both
    for label, df in [('Scopus', scopus), ('Abstracts', abstracts)]:
        if 'EID' not in df.columns:
            raise ValueError(f"{inst_name} {label} file has no EID column. Columns: {df.columns.tolist()}")

    # Identify the abstract column in the abstracts file
    abstract_col = next(
        (c for c in abstracts.columns if 'abstract' in c.lower()),
        None
    )
    if abstract_col is None:
        print(f"  WARNING: no 'Abstract' column found in abstracts file.")
        print(f"  Available columns: {abstracts.columns.tolist()}")
        return None

    print(f"  Abstract column in abstracts file: '{abstract_col}'")

    # If scopus already has an Abstract column, drop it (it's empty/URL junk)
    if 'Abstract' in scopus.columns:
        print(f"  Dropping existing Abstract column from scopus (will be replaced)")
        scopus = scopus.drop(columns=['Abstract'])

    # Pull only EID + abstract from abstracts file
    abs_subset = abstracts[['EID', abstract_col]].copy()
    abs_subset = abs_subset.rename(columns={abstract_col: 'Abstract'})
    abs_subset = abs_subset.drop_duplicates(subset='EID')

    # Left join: keep all scopus rows, bring in abstract where available
    merged = scopus.merge(abs_subset, on='EID', how='left')

    matched = merged['Abstract'].notna().sum()
    print(f"  Merged:    {merged.shape[0]:,} rows × {merged.shape[1]} cols")
    print(f"  Abstracts matched: {matched:,} / {len(merged):,} ({matched/len(merged)*100:.1f}%)")
    print(f"  EIDs with no abstract: {len(merged) - matched:,}")

    # Tag institution
    merged['institution'] = inst_name

    return merged


institution_dfs = {}

for inst, (scopus_path, abstracts_path) in INSTITUTIONS.items():
    if not Path(scopus_path).exists() or not Path(abstracts_path).exists():
        print(f"SKIPPING {inst} — one or both files missing")
        continue
    df = merge_institution(inst, scopus_path, abstracts_path)
    if df is not None:
        institution_dfs[inst] = df

print(f"\nSuccessfully merged: {list(institution_dfs.keys())}")

## 3. Inspect columns per institution before combining

In [ ]:
all_cols = sorted(set(c for df in institution_dfs.values() for c in df.columns))
col_matrix = pd.DataFrame(
    {inst: [c in df.columns for c in all_cols] for inst, df in institution_dfs.items()},
    index=all_cols
)
col_matrix['in_all'] = col_matrix.all(axis=1)
col_matrix['count']  = col_matrix.drop(columns='in_all').sum(axis=1)

print(f"Columns present in ALL institutions: {col_matrix['in_all'].sum()} / {len(all_cols)}")
print()
col_matrix.sort_values('count', ascending=False)

In [ ]:
# Columns present in some but not all
partial = col_matrix[~col_matrix['in_all']].drop(columns='in_all').sort_values('count', ascending=False)
if partial.empty:
    print("All columns are shared across institutions.")
else:
    print(f"Columns NOT shared by all ({len(partial)} total):")
    partial

## 4. Combine all institutions

In [ ]:
combined = pd.concat(institution_dfs.values(), ignore_index=True)

print(f"Combined shape: {combined.shape[0]:,} rows × {combined.shape[1]} cols")
print(f"\nRows per institution:")
print(combined['institution'].value_counts().to_string())

## 5. Cross-institution EID duplicates

In [ ]:
dup_eids = combined[combined.duplicated(subset='EID', keep=False)]
print(f"Rows with a duplicated EID (cross-institution): {len(dup_eids):,}")

if len(dup_eids) > 0:
    print(f"Unique EIDs appearing in >1 institution: {dup_eids['EID'].nunique():,}")
    print()
    print("Institution breakdown of duplicated EIDs:")
    print(dup_eids['institution'].value_counts().to_string())
    print()
    print("Sample duplicated EIDs:")
    dup_eids[['EID', 'institution']].sort_values('EID').head(10)

In [ ]:
# De-duplicate: keep first occurrence, add comma-separated list of all institutions
# where the EID appeared
inst_per_eid = (
    combined.groupby('EID')['institution']
    .apply(lambda x: ','.join(sorted(set(x))))
    .reset_index()
    .rename(columns={'institution': 'institutions'})
)

combined_dedup = (
    combined
    .drop_duplicates(subset='EID', keep='first')
    .merge(inst_per_eid, on='EID', how='left')
)

print(f"Before dedup: {len(combined):,} rows")
print(f"After dedup:  {len(combined_dedup):,} rows")
print(f"Removed:      {len(combined) - len(combined_dedup):,} cross-institution duplicates")

## 6. Post-merge sanity check

In [ ]:
key_cols = ['EID', 'Abstract', 'Citations', 'Year', 'Authors', 'Title', 'Source title']

print(f"{'Column':<20} {'Present':<10} {'Missing':>10} {'Missing %':>12}")
print("-" * 55)
for col in key_cols:
    if col in combined_dedup.columns:
        n_missing = combined_dedup[col].isna().sum()
        pct = n_missing / len(combined_dedup) * 100
        print(f"{col:<20} {'✓':<10} {n_missing:>10,} {pct:>11.1f}%")
    else:
        print(f"{col:<20} {'✗ MISSING':<10}")

In [ ]:
combined_dedup.head(3)

## 7. Save

In [ ]:
out_dir = Path('../data/processed')
out_dir.mkdir(parents=True, exist_ok=True)

out_csv = out_dir / 'peer_unis_merged.csv'
out_pkl = out_dir / 'peer_unis_merged.pkl'

combined_dedup.to_csv(out_csv, index=False)
combined_dedup.to_pickle(out_pkl)

print(f"Saved {len(combined_dedup):,} rows to:")
print(f"  {out_csv}")
print(f"  {out_pkl}")